In [ ]:
# %% ── 1. Imports and paths ───────────────────────────────────────────────────
 
from __future__ import annotations
 
import sys
import json
import warnings
from pathlib import Path
 
import geopandas as gpd
import numpy as np
import pandas as pd
import shapely
from shapely import STRtree
from shapely.geometry import Point, Polygon
import folium
from folium.features import GeoJsonTooltip
 
# ── Project root and src on path ─────────────────────────────────────────────
PROJECT_ROOT = Path(
    "/Users/jedrek/Documents/Studium Volkswirschaftslehre/"
    "4. Semester/DEDA Project/DEDA_LLM_Spatial_Hotelling"
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
 
from hotelling.spatial.osm import fetch_pois     # uses project Overpass infra
 
PATH_RAW       = PROJECT_ROOT / "data" / "raw"
PATH_PROCESSED = PROJECT_ROOT / "data" / "processed"
ANALYSIS_CRS   = "EPSG:3035"    # equal-area metric; all distance / area work here
DISPLAY_CRS    = "EPSG:4326"    # WGS-84 for Folium

In [ ]:
# Load all inputs required by build_commercial_candidates.
# These files must already exist on disk (produced by earlier GEO notebooks
# and download functions).

from pathlib import Path

# Inner-Ringbahn boundary polygon.
# relation_boundary_14983.geojson stores EPSG:3035 coords but declares EPSG:4326
# (known CRS mismatch) — detect by coordinate magnitude and override.
boundary = gpd.read_file(PATH_RAW / "relation_boundary_14983.geojson")
if boundary.total_bounds[0] > 1000:   # metric coords → EPSG:3035, not EPSG:4326
    boundary = boundary.set_crs(ANALYSIS_CRS, allow_override=True)
else:
    boundary = boundary.to_crs(ANALYSIS_CRS)

# Active incumbents (for exclusion)
incumbents = gpd.read_parquet(
    PATH_RAW / "OSM_POIs_Berlin_supermarket.parquet"
).to_crs(ANALYSIS_CRS)

# BRW land values
brw = gpd.read_file(PATH_RAW / "brw_2025.gpkg").to_crs(ANALYSIS_CRS)

# Stadtstruktur morphology zones
stadtstruktur = gpd.read_file(PATH_RAW / "stadtstruktur.gpkg").to_crs(ANALYSIS_CRS)

print("Inputs loaded.")
print(f"  boundary: {boundary.geometry.geom_type.values[0]}, CRS={boundary.crs.to_epsg()}")

In [ ]:
from hotelling.spatial.parcels import build_commercial_candidates

L_commercial_final = build_commercial_candidates(
    boundary=boundary,
    alkis_path=PATH_RAW / "alkis_full.gpkg",
    incumbents=incumbents,
    brw=brw,
    stadtstruktur=stadtstruktur,
    # Filtering thresholds (package defaults match GEO_06 notebook):
    min_footprint_m2=400.0,
    min_mbr_dim_m=10.0,
    max_mbr_aspect=10.0,
    incumbent_exclusion_buffer_m=50.0,
    save_path=PATH_PROCESSED,
)

print(f"\nL_commercial_final: {len(L_commercial_final)} candidate locations")
print(f"  Tier A: {(L_commercial_final['confidence_tier']=='A').sum()}")
print(f"  Tier B: {(L_commercial_final['confidence_tier']=='B').sum()}")
print(L_commercial_final["shop_type"].value_counts().to_string())

In [ ]:
# %% ── 13. Interactive HTML map ───────────────────────────────────────────────
 
_TIER_COLOR = {"A": "#c0392b", "B": "#e67e22"}
_TIER_LABEL = {
    "A": "A — Former grocery / wholesale",
    "B": "B — Former large-format non-grocery retail",
}
 
# Reproject to WGS-84 for Folium
_L_4326 = L_commercial_final.drop(columns=["centroid"], errors="ignore").to_crs(DISPLAY_CRS).copy()
# Simplify polygons for performance (1 m tolerance in metric CRS → reproject)
_L_simplified = (
    L_commercial_final.drop(columns=["centroid"], errors="ignore")
    .to_crs("EPSG:3035")
    .assign(geometry=lambda g: g.geometry.simplify(1.0, preserve_topology=True))
    .to_crs(DISPLAY_CRS)
)
# Ensure bool columns are serialisable
for _c in _L_simplified.select_dtypes(include=["bool"]).columns:
    _L_simplified[_c] = _L_simplified[_c].astype(object)
 
_center_y = _L_4326.geometry.centroid.y.mean()
_center_x = _L_4326.geometry.centroid.x.mean()
 
m = folium.Map(
    location=[_center_y, _center_x],
    zoom_start=12,
    tiles="OpenStreetMap",
    prefer_canvas=True,
)
 
# Ringbahn boundary
folium.GeoJson(
    boundary.to_crs(DISPLAY_CRS).__geo_interface__,
    name="Inner Ringbahn",
    style_function=lambda x: {
        "fillColor": "none", "color": "#8e44ad",
        "weight": 2.0, "dashArray": "6 4", "fillOpacity": 0,
    },
    tooltip="Inner Ringbahn boundary",
).add_to(m)
 
# One layer per tier
_TOOLTIP_COLS = [c for c in [
    "id", "confidence_tier", "shop_type", "shop_signal",
    "footprint_m2", "mbr_min_dim_m", "brw_value",
    "typ_klar", "morphology_score",
] if c in _L_simplified.columns]
 
for _tier in ["A", "B"]:
    _sub = _L_simplified[_L_simplified["confidence_tier"] == _tier].copy()
    if _sub.empty:
        continue
    _color = _TIER_COLOR[_tier]
    folium.GeoJson(
        data=_sub[_TOOLTIP_COLS + ["geometry"]],
        name=f"Tier {_TIER_LABEL[_tier]}",
        style_function=lambda x, c=_color: {
            "fillColor": c, "color": c,
            "weight": 0.6, "fillOpacity": 0.65,
        },
        highlight_function=lambda x: {"weight": 2.5, "fillOpacity": 0.90},
        tooltip=GeoJsonTooltip(
            fields=_TOOLTIP_COLS,
            aliases=[c.replace("_", " ").title() for c in _TOOLTIP_COLS],
            localize=True, sticky=False,
        ),
    ).add_to(m)
 
folium.LayerControl(collapsed=False).add_to(m)
 
# Legend
_tier_counts = L_commercial_final["confidence_tier"].value_counts()
_legend_rows = "".join(
    f'<div style="display:flex;align-items:center;margin-bottom:5px;">'
    f'<div style="width:16px;height:16px;background:{_TIER_COLOR[t]};opacity:0.75;'
    f'border:1px solid #555;margin-right:8px;flex-shrink:0;"></div>'
    f'<span>{_TIER_LABEL[t]}&nbsp;<b>({_tier_counts.get(t, 0):,})</b></span></div>'
    for t in ["A", "B"]
)
m.get_root().html.add_child(folium.Element(f"""
<div style="position:fixed;bottom:40px;left:40px;z-index:9999;
            background:rgba(255,255,255,0.93);padding:12px 16px;
            border-radius:8px;border:1px solid #aaa;
            font-family:sans-serif;font-size:12px;
            box-shadow:2px 2px 6px rgba(0,0,0,0.2);min-width:280px;">
  <div style="font-weight:bold;margin-bottom:8px;font-size:13px;">
    𝓛<sup>commercial</sup> — disused large-format retail
    &nbsp;<span style="font-weight:normal;">({len(L_commercial_final):,} locations)</span>
  </div>
  {_legend_rows}
  <hr style="border:none;border-top:1px solid #ddd;margin:8px 0;">
  <div style="color:#555;font-size:11px;">
    Polygon = building footprint (ALKIS or OSM).<br>
    Hover over polygon for attributes.
  </div>
</div>
"""))
 
_map_path = PATH_PROCESSED / "L_commercial_final_map.html"
m.save(str(_map_path))
print(f"Map saved → {_map_path}  ({_map_path.stat().st_size / 1e6:.1f} MB)")